# Week 3, day 4 (morning) — Worksheet 08 SOLUTIONS: loading the dimension tables

Executed in the lab image. Every quoted number is what it actually printed.

Question 7 is the one that saves you later. An `Unknown` member costs one row per
dimension and it is what stops worksheet 09's fact load from silently dropping
rows.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 08 — Loading the dimension tables. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr, tx = load("enrollment"), load("transaction")
crs, prg, cat = load("course"), load("program"), load("category")
coh, stu, cty = load("cohort"), load("students"), load("city")
dtype = load("discount_type")


def add_surrogate_key(df, name, start=1):
    """Assign a surrogate key 1..n, ordered by the source key for stability."""
    out = df.reset_index(drop=True).copy()
    out.insert(0, name, range(start, start + len(out)))
    return out


print("source tables loaded for six dimensions")

PART A — the straightforward dimensions

### Question 1

Build `dim_course` to slide 29's column list: a surrogate `course_id`, `source_course_id`, `course_name`, `credit_hours`, `is_active`. Print the shape, the columns, and the first three rows.

In [ ]:
dim_course = (crs[["course_id", "course_name", "hours", "active_flg"]]
              .rename(columns={"course_id": "source_course_id",
                               "hours": "credit_hours",
                               "active_flg": "is_active"})
              .sort_values("source_course_id"))
dim_course = add_surrogate_key(dim_course, "course_id")

print("dim_course:", dim_course.shape)
print("columns:", list(dim_course.columns))
print()
print(dim_course.head(3).to_string(index=False))
print()
print("surrogate key unique:", dim_course.course_id.is_unique)
print("source key unique:   ", dim_course.source_course_id.is_unique)

```
dim_course: (24, 5)
columns: ['course_id', 'source_course_id', 'course_name', 'credit_hours', 'is_active']

 course_id  source_course_id                  course_name  credit_hours  is_active
         1               201       SQL for Data Engineers            42          1
         2               202 Data Modeling and ETL Design            30          1
         3               203  Python for Data Engineering            42          1

surrogate key unique: True
source key unique:    True
```

Twenty-four rows, and the whole load is a select, a rename and a key.

Slide 38's four steps map onto four lines of code here: *select descriptive
attributes* is the column list, *remove duplicates* is unnecessary because
`course_id` is already unique, *assign dimension keys* is `add_surrogate_key`,
and *organize into meaningful tables* is the rename.

The renames are the part worth noticing. `hours` becomes `credit_hours` and
`active_flg` becomes `is_active` — both because the target name says what the
column means without needing the source system's conventions. A warehouse is read
by people who will never see the source ER diagram.

**Both uniqueness checks pass, and both are worth printing.** `course_id` unique
is a property of the generator, so checking it only proves the code works.
`source_course_id` unique is a property of the *source*, and it is the one that
can fail — a source extract with a duplicated key produces a dimension with a
duplicated business key, and question 10 shows what that does to the fact table.

`is_active` carries the one retired course from worksheet 01 question 9. Note it
is kept, not filtered: a course that is no longer offered still has historical
enrollments, and deleting its dimension row would orphan them. **Dimensions
record what existed, not what exists.**

### Question 2

Build `dim_cohort` the same way. Then show why the surrogate key must be assigned in a **stable** order: rebuild it from the source sorted differently, and compare the keys the two builds give to cohort 305.
> **NOTE:** if last night's fact rows point at `cohort_id = 5`, tonight's rebuild had better give the same cohort that key.

In [ ]:
base = coh[["cohort_id", "cohort_name", "start_dt", "end_dt"]].rename(
    columns={"cohort_id": "source_cohort_id", "start_dt": "start_date",
             "end_dt": "end_date"})

stable = add_surrogate_key(base.sort_values("source_cohort_id"), "cohort_id")
shuffled = add_surrogate_key(
    base.sort_values("start_date", ascending=False), "cohort_id")

print("dim_cohort:", stable.shape)
print(stable.head(3).to_string(index=False))
print()
for label, df in [("build A: by source key ", stable),
                  ("build B: by start_date  ", shuffled)]:
    print("  %s" % label)
    for src in (301, 305, 316):
        k = int(df.loc[df.source_cohort_id == src, "cohort_id"].iloc[0])
        print("      source cohort %d -> surrogate key %2d" % (src, k))
same = (stable.sort_values("source_cohort_id").cohort_id.tolist()
        == shuffled.sort_values("source_cohort_id").cohort_id.tolist())
print()
print("  the two builds agree on every key:", same)

```
  build A: by source key
      source cohort 301 -> surrogate key  1
      source cohort 305 -> surrogate key  5
      source cohort 316 -> surrogate key 16
  build B: by start_date
      source cohort 301 -> surrogate key 16
      source cohort 305 -> surrogate key 12
      source cohort 316 -> surrogate key  1

  the two builds agree on every key: False
```

Same source data, same code, one different `ORDER BY` — and **cohort 301 is key
1 in one build and key 16 in the other.**

Now imagine that as two consecutive nightly runs. Last night `fact_enrollment`
recorded 200 enrollments against `cohort_id = 1`. Tonight the dimension is
rebuilt with a different sort, and `cohort_id = 1` is now a completely different
cohort. **Every historical fact row silently points at the wrong cohort.**

Nothing errors. Every foreign key still resolves — key 1 exists, it just means
something else now. Row counts are unchanged, referential integrity passes, and
every report by cohort is wrong in a way no check in worksheet 10 would catch.

This is the failure mode surrogate keys are famous for, and it has one rule:

> **A surrogate key, once assigned, is permanent.**

Three ways to guarantee that, in increasing order of robustness:

**Sort deterministically by the source key**, as build A does. Adequate when the
dimension is fully rebuilt each run and no rows are ever removed — but a source
row disappearing shifts every key after it.

**Never rebuild — only append.** Keep the dimension, look up existing source keys,
and assign new surrogates only to genuinely new rows. This is what real dimension
loads do, and it makes the key permanent by construction.

**Use a database sequence or identity column**, which does the same thing and
survives the code being rewritten.

The reason to prefer the second or third is that build A's stability is an
*accident of the sort order* rather than a guarantee. It holds until someone
changes an `ORDER BY` for an unrelated reason — which is exactly the change that
produced build B.

### Question 3

Build `dim_program`, which needs `program_category` from the `category` table. Apply worksheet 06's rule R4, and print the result in full.

In [ ]:
dim_program = (prg[["program_id", "program_name", "category_id", "active_flg"]]
               .merge(cat[["category_id", "category_name"]],
                      on="category_id", how="left"))
dim_program["category_name"] = dim_program["category_name"].fillna("Unknown")
dim_program = (dim_program
               .rename(columns={"program_id": "source_program_id",
                                "category_name": "program_category",
                                "active_flg": "is_active"})
               .drop(columns=["category_id"])
               .sort_values("source_program_id"))
dim_program = add_surrogate_key(dim_program, "program_id")

print("dim_program:", dim_program.shape)
print()
print(dim_program.to_string(index=False))

```
dim_program: (8, 5)

 program_id  source_program_id                 program_name  is_active program_category
          1                101     Applied Data Engineering          1 Data Engineering
          2                102        Analytics Engineering          1 Data Engineering
          3                103 Machine Learning Foundations          1     Data Science
          4                104         Applied Data Science          1     Data Science
          5                105   Cloud Platform Engineering          1  Cloud Computing
          6                106 Site Reliability Engineering          0  Cloud Computing
          7                107          Security Operations          1    Cybersecurity
          8                108         Foundations Bootcamp          1          Unknown
```

Eight programs, and the whole dimension fits on one screen — which is worth
pausing on, because this table is the reason for three joins in the source and
one in the star.

**Row 8 is where rule R4 lands.** `Foundations Bootcamp` points at category 5,
whose name is null, and it now reads `Unknown` rather than blank. Those 316
enrollments from worksheet 06 question 4 will group under a visible label in every
report built on this dimension.

**Row 6 is `is_active = 0`** — Site Reliability Engineering, retired. Kept for the
same reason as the retired course: its enrollments are real history.

Note that `program_category` is denormalised in — the category table does not
appear in the target model at all. That is worksheet 04's star decision, applied:
five category names stored as eight copies, in exchange for one join instead of
two. At this scale it is unambiguously the right call.

One thing this dimension cannot do, and it is worth knowing the limit: if
`Foundations Bootcamp` moves from category 5 to category 2 next month, a rebuild
overwrites row 8 and **every historical enrollment retroactively changes
category**. Last quarter's report will not reproduce. That is a Type 1 slowly
changing dimension, and it is the default behaviour of every load in this
worksheet. Preserving history needs a Type 2 — a second row with validity dates,
which is only possible because the surrogate key is separate from the source key.

### Question 4

Build `dim_student`, which denormalizes `city`, `state` and `country` from the `city` table. Print the shape, check the row count did not change, and count how many students end up with no city.
> **NOTE:** worksheet 01 question 9 found students with a null `city_id`. Decide what they get.

In [ ]:
dim_student = (stu[["stu_id", "stu_name", "birthday", "city_id", "active_flg"]]
               .merge(cty[["city_id", "city_name", "provn_name", "cntry_name"]],
                      on="city_id", how="left"))
print("students in:", len(stu), "-> after city join:", len(dim_student))

dim_student = (dim_student
               .rename(columns={"stu_id": "source_student_id",
                                "stu_name": "student_name",
                                "birthday": "date_of_birth",
                                "city_name": "city",
                                "provn_name": "state",
                                "cntry_name": "country",
                                "active_flg": "is_active"})
               .drop(columns=["city_id"])
               .sort_values("source_student_id"))
for col in ("city", "state", "country"):
    print("  null %-8s before default: %d" % (col, int(dim_student[col].isna().sum())))
    dim_student[col] = dim_student[col].fillna("Unknown")
dim_student = add_surrogate_key(dim_student, "student_id")

print()
print("dim_student:", dim_student.shape)
print(dim_student.head(3).to_string(index=False))

```
students in: 606 -> after city join: 606
  null city     before default: 7
  null state    before default: 7
  null country  before default: 7

dim_student: (606, 8)
 student_id  source_student_id    student_name date_of_birth  is_active     city            state       country
          1               5001 Ursula Quintero    1986-10-05          1  Unknown          Unknown       Unknown
          2               5002  Elena Ferreira    1994-08-06          1 Victoria British Columbia        Canada
          3               5003  Kenji Kowalski    1988-06-18          1  Seattle       Washington United States
```

606 in, 606 out — the row count check that confirms the `city` join was
many-to-one and did not fan out.

**Seven students have no city**, and all three geography columns are null for the
same seven — they are one fact, not three. The `Unknown` default applies to all
three, so a report grouped by country shows `Unknown: 7` rather than silently
losing them. Same argument as worksheet 06 question 4.

The geography is **denormalised**: `city`, `state`, `country` as names, side by
side, rather than a `city_id` pointing at a city dimension. This is the star
schema decision again, and here it is doing more work than in `dim_program`. A
snowflake would need `dim_student` → `dim_city` → `dim_province` → `dim_country`,
which is three extra joins for *"enrollments by country"* — one of the more
obvious questions anyone will ask.

Note what the source made easy: `city` already carried `provn_name` and
`cntry_name` inline (worksheet 01 question 8). The operational schema had already
made the same trade-off, so this is a copy rather than a flattening.

Two things this dimension should probably also have, and does not because slide 29
does not list them:

**An age band rather than `date_of_birth`.** A raw birth date is personal data with
a compliance cost, and almost every analytical use of it is "which age group". A
derived band is more useful and less sensitive.

**A `_loaded_at` column**, so you can tell when a dimension row last changed. With
Type 1 overwrites and no history, that timestamp is the only trace of a change
having happened at all.

PART B — the dimension with no source

### Question 5

Build `dim_date`. There is no source table, so generate one row per calendar day covering the full range of `enrl_date`, with all of slide 29's columns. Print the shape, the range, and three rows.
> **NOTE:** generate every day in the range, not only the days that appear. A date dimension with gaps cannot answer "which days had no enrollments".

In [ ]:
dates = pd.to_datetime(enr.enrl_date)
full = pd.date_range(dates.min(), dates.max(), freq="D")

dim_date = pd.DataFrame({"full_date": full})
dim_date["date_id"] = dim_date.full_date.dt.strftime("%Y%m%d").astype(int)
dim_date["day_of_week"] = dim_date.full_date.dt.day_name()
dim_date["day"] = dim_date.full_date.dt.day
dim_date["week"] = dim_date.full_date.dt.isocalendar().week.astype(int)
dim_date["month"] = dim_date.full_date.dt.month
dim_date["quarter"] = dim_date.full_date.dt.quarter
dim_date["year"] = dim_date.full_date.dt.year
dim_date["is_weekend"] = dim_date.full_date.dt.dayofweek.isin([5, 6]).astype(int)
dim_date = dim_date[["date_id", "full_date", "day_of_week", "day", "week",
                     "month", "quarter", "year", "is_weekend"]]

print("dim_date:", dim_date.shape)
print("range:", full.min().date(), "to", full.max().date())
print("distinct enrollment dates in the source:", dates.nunique())
print("days with no enrollment:", len(full) - dates.nunique())
print()
print(dim_date.head(3).to_string(index=False))

```
dim_date: (675, 9)
range: 2023-11-24 to 2025-09-28
distinct enrollment dates in the source: 650
days with no enrollment: 25

 date_id  full_date day_of_week  day  week  month  quarter  year  is_weekend
20231124 2023-11-24      Friday   24    47     11        4  2023           0
20231125 2023-11-25    Saturday   25    47     11        4  2023           1
20231126 2023-11-26      Sunday   26    47     11        4  2023           1
```

**675 rows for 650 distinct enrollment dates.** The 25 extra are the days nothing
happened, and generating them is the entire point of the question.

A date dimension built from `SELECT DISTINCT enrl_date` would have 650 rows and
would be unable to answer *"which days had no enrollments?"* — because those days
would not exist in the model. Days with zero activity are data. A time series with
missing rows silently becomes a time series with the gaps closed up, and every
chart of it is subtly wrong.

`dim_date` is the only table here with no source, and it is the most useful table
in most warehouses, for one reason: **it turns date logic into joins.** Instead of
every query computing `EXTRACT(QUARTER FROM ...)` or wrestling with week
numbering, it joins to a table that already has `quarter` and `week` as columns.
Every query agrees on what week 47 means, because there is one row saying so.

Three details in the implementation.

**`date_id` is `YYYYMMDD` as an integer**, not a sequence. This is the standard
exception to "surrogate keys should be meaningless": a date key is naturally
unique, naturally ordered, human-readable in a query result, and stable across
rebuilds forever. It also makes partition pruning trivial.

**`is_weekend` is precomputed.** It costs one column and removes a `CASE`
expression from every query that needs it — and removes the chance of two analysts
disagreeing about whether Saturday counts.

**The range is driven by the facts.** Extending it is normal practice: most
warehouses generate the date dimension years ahead, so a late-arriving fact never
finds its date missing. Note that this build would need re-running when data
arrives past 2025-09-28, which is exactly the sort of dependency worth generating
generously rather than exactly.

### Question 6

Build `dim_promotion` from `discount_type`, applying rule R3 to the null amount. Print it in full.

In [ ]:
dim_promotion = (dtype[["discount_type_id", "discount_type_name",
                        "discount_amount"]]
                 .rename(columns={"discount_type_id": "source_discount_type_id",
                                  "discount_type_name": "promotion_name"})
                 .sort_values("source_discount_type_id"))
print("null discount_amount before rule R3:",
      int(dim_promotion.discount_amount.isna().sum()))
dim_promotion["discount_amount"] = dim_promotion["discount_amount"].fillna(0.0)
dim_promotion = add_surrogate_key(dim_promotion, "promotion_id")

print()
print(dim_promotion.to_string(index=False))

```
null discount_amount before rule R3: 1

 promotion_id  source_discount_type_id   promotion_name  discount_amount
            1                       10       Early Bird            500.0
            2                       20  Alumni Referral            750.0
            3                       30 Employer Partner           1200.0
            4                       40      Scholarship           2000.0
            5                       50  Spring Campaign            300.0
            6                       60 Legacy Promotion              0.0
```

Six promotions, and `Legacy Promotion` now reads 0.00 instead of null.

Worth being precise about what that zero is claiming. Worksheet 06 question 3
separated two causes of a null discount, and this is the second one: the promotion
was applied to **197 enrollments** and nobody recorded what it was worth. The zero
is not a measurement; it is a decision to treat unknown as nothing.

That decision understates discounting by whatever those 197 promotions were
actually worth, and the dimension row is now the only place it is visible — as a
promotion with a name and no value, sitting in a table of five promotions that all
have one. Anyone who reads `dim_promotion` will notice. Nobody reading a discount
total ever would.

Two improvements worth making in a real load:

**Carry a flag.** `discount_is_estimated` on the fact row, or `amount_is_known` on
the dimension, so a discount-rate calculation can report how much of it rests on
an assumption.

**Escalate it.** 197 enrollments with an unpriced promotion is a five-minute
question for whoever maintains the promotions table, and the answer exists.
Defaulting to zero is the right *interim* behaviour; it is a poor permanent one.

Note this dimension has a genuine one-to-one relationship with its source — six
rows in, six out — so the load is a rename and a key. `discount_id`, the source's
own surrogate, is dropped: the model keys on `discount_type_id`, which is what
`transaction` actually references. **Map the key the facts use, not the one the
source table happens to have as its PK.**

PART C — the Unknown member

### Question 7

Every dimension needs a row for facts that match nothing. Add an `Unknown` member with key `-1` to `dim_student` and `dim_promotion`, and print the new row of each.
> **NOTE:** worksheet 05 question 6 found 7 enrollments whose student does not exist, and worksheet 06 question 3 found 1,019 with no promotion at all. Both need somewhere to point.

In [ ]:
dim_student = (stu[["stu_id", "stu_name", "birthday", "active_flg"]]
               .rename(columns={"stu_id": "source_student_id",
                                "stu_name": "student_name",
                                "birthday": "date_of_birth",
                                "active_flg": "is_active"})
               .sort_values("source_student_id"))
dim_student = add_surrogate_key(dim_student, "student_id")

unknown_student = pd.DataFrame([{
    "student_id": -1, "source_student_id": -1,
    "student_name": "Unknown", "date_of_birth": None, "is_active": 0}])
dim_student = pd.concat([unknown_student, dim_student], ignore_index=True)

dim_promotion = pd.DataFrame([
    {"promotion_id": -1, "source_discount_type_id": -1,
     "promotion_name": "No Promotion", "discount_amount": 0.0}])

print("dim_student, first two rows:")
print(dim_student.head(2).to_string(index=False))
print()
print("dim_promotion Unknown member:")
print(dim_promotion.to_string(index=False))
print()
print("rows that will point at student -1:  ",
      int((~enr.stu_id.isin(set(stu.stu_id))).sum()))
print("rows that will point at promotion -1:",
      int(tx.groupby("enrl_id").discount_type_id.max().isna().sum()))

```
dim_student, first two rows:
 student_id  source_student_id    student_name date_of_birth  is_active
         -1                 -1         Unknown          None          0
          1               5001 Ursula Quintero    1986-10-05          1

dim_promotion Unknown member:
 promotion_id  source_discount_type_id promotion_name  discount_amount
           -1                       -1   No Promotion              0.0

rows that will point at student -1:   7
rows that will point at promotion -1: 1019
```

One extra row per dimension, and it is the difference between a fact load that
drops rows and one that does not.

**1,019 enrollments have no promotion.** Without a member to point at, their
`promotion_id` is null — and a null foreign key means every query joining
`fact_enrollment` to `dim_promotion` silently loses 45% of the fact table unless
it remembers `LEFT JOIN`. With `promotion_id = -1`, an ordinary inner join keeps
them and they group under `No Promotion`.

**7 enrollments reference a student who does not exist.** Worksheet 06 question 5
argued against excluding these, and this is where that argument pays: they land on
student `-1`, stay countable, and stay visible as an anomaly somebody can chase.

Three conventions that make this work.

**Negative keys.** `-1` cannot collide with a generated sequence starting at 1,
and it is instantly recognisable in a query result. Some shops use `0`; the only
requirement is that it is reserved.

**A meaningful label.** `No Promotion` and `Unknown` say different things, and the
difference matters. `No Promotion` is a **business fact** — the enrollment
genuinely had none. `Unknown` is a **data problem** — we should know and do not.
Using one label for both hides which is which. Where both occur in one dimension,
two members (`-1` Unknown, `-2` Not Applicable) is the honest answer.

**`is_active = 0`** on the Unknown member, so it does not appear in dimension
browsers or picklists as if it were a real student.

The habit: **add the Unknown member when you create the dimension, not when a
load fails.** It costs one row and it converts the most common fact-load failure
from a silent drop into a visible category.

### Question 8

Check every dimension before the fact load. For each of the six, print the row count, whether the surrogate key is unique, whether the source key is unique, and the null count across all columns.
> **NOTE:** a non-unique dimension key is what silently multiplies fact rows. Check it here, where it is cheap.

In [ ]:
dims = {}
dims["dim_course"] = add_surrogate_key(
    crs[["course_id"]].rename(columns={"course_id": "source_course_id"})
       .sort_values("source_course_id"), "course_id")
dims["dim_program"] = add_surrogate_key(
    prg[["program_id"]].rename(columns={"program_id": "source_program_id"})
       .sort_values("source_program_id"), "program_id")
dims["dim_cohort"] = add_surrogate_key(
    coh[["cohort_id"]].rename(columns={"cohort_id": "source_cohort_id"})
       .sort_values("source_cohort_id"), "cohort_id")
dims["dim_student"] = add_surrogate_key(
    stu[["stu_id"]].rename(columns={"stu_id": "source_student_id"})
       .sort_values("source_student_id"), "student_id")
dims["dim_promotion"] = add_surrogate_key(
    dtype[["discount_type_id"]]
    .rename(columns={"discount_type_id": "source_discount_type_id"})
    .sort_values("source_discount_type_id"), "promotion_id")
d = pd.to_datetime(enr.enrl_date)
dims["dim_date"] = pd.DataFrame({
    "date_id": pd.date_range(d.min(), d.max(), freq="D").strftime("%Y%m%d").astype(int)})

print("%-15s %6s %10s %10s %6s" % ("DIMENSION", "ROWS", "PK UNIQUE", "SRC UNIQUE", "NULLS"))
for name, df in dims.items():
    pk = df.columns[0]
    src = df.columns[1] if len(df.columns) > 1 else pk
    print("%-15s %6d %10s %10s %6d"
          % (name, len(df), df[pk].is_unique, df[src].is_unique,
             int(df.isna().sum().sum())))

```
DIMENSION         ROWS  PK UNIQUE SRC UNIQUE  NULLS
dim_course          24       True       True      0
dim_program          8       True       True      0
dim_cohort          16       True       True      0
dim_student        606       True       True      0
dim_promotion        6       True       True      0
dim_date           675       True       True      0
```

Six dimensions, all clean, and this table is the gate before the fact load.

**`PK UNIQUE` proves the generator worked.** Useful, but it is checking your own
code — it cannot fail unless `add_surrogate_key` is broken.

**`SRC UNIQUE` is the check that matters.** It is a property of the *source*, and
it is what makes the fact table's key lookup a one-to-one mapping. When it fails —
a source extract that double-loaded, a merged system with colliding ids — the
lookup returns two rows for one fact, and the fact table grows. Question 10 shows
that at 2,483 rows instead of 2,400.

**`NULLS` catches a rule that did not run.** Zero here means R3 and R4 fired and
the `Unknown` defaults were applied. A non-zero would mean a dimension attribute
that will show as blank in every report.

Running these six checks before loading facts is the cheap version of worksheet
10's referential integrity check. The difference is *when* you find out: a
dimension check fails in wave 2 with six small tables and an obvious cause; the
same problem found after the fact load means diagnosing a 2,400-row table and
re-running everything.

Two checks worth adding that this table does not do:

**Row count against the previous run.** A dimension that drops from 606 students
to 4 is a broken extract, and no uniqueness check will notice.

**No unexpected `Unknown`.** If `dim_student.city` is `Unknown` for 400 rows
instead of 7, the city join has broken. The count of defaulted values is a
monitorable number — which is exactly what worksheet 06's rule register was for.

### Question 9

Summarise the load. Print each dimension's row count against the source table it came from, and the total rows across all six.

In [ ]:
d = pd.to_datetime(enr.enrl_date)
n_dates = len(pd.date_range(d.min(), d.max(), freq="D"))
rows = [
    ("dim_course", "course", len(crs), len(crs)),
    ("dim_program", "program", len(prg), len(prg)),
    ("dim_cohort", "cohort", len(coh), len(coh)),
    ("dim_student", "students", len(stu), len(stu) + 1),
    ("dim_promotion", "discount_type", len(dtype), len(dtype) + 1),
    ("dim_date", "(generated)", 0, n_dates),
]
print("%-15s %-14s %8s %8s" % ("DIMENSION", "SOURCE", "SRC ROWS", "DIM ROWS"))
for name, src, a, b in rows:
    print("%-15s %-14s %8d %8d" % (name, src, a, b))
print()
print("total dimension rows:", sum(r[3] for r in rows))
print("(dim_student and dim_promotion each carry one Unknown member)")

```
DIMENSION       SOURCE         SRC ROWS DIM ROWS
dim_course      course               24       24
dim_program     program               8        8
dim_cohort      cohort               16       16
dim_student     students            606      607
dim_promotion   discount_type         6        7
dim_date        (generated)           0      675

total dimension rows: 1337
(dim_student and dim_promotion each carry one Unknown member)
```

**1,337 dimension rows against a 2,400-row fact table** — and that ratio is the
shape of a healthy star schema. Dimensions are small; facts accumulate. In a year
this fact table is 30,000 rows and the dimensions are still roughly 1,400.

Read the source-to-dimension column as an audit. Four dimensions match their
source exactly, which is what you want when the load is a straight copy — any
difference there would mean rows were dropped or duplicated.

The three that differ, differ for stated reasons:

**`dim_student` 606 → 607** and **`dim_promotion` 6 → 7**: the Unknown members from
question 7. Deliberate, documented, one row each.

**`dim_date` 0 → 675**: no source at all. It is generated from the fact date
range, and it is the largest dimension in the model — larger than `dim_student`.
That is normal and it is why date dimensions are usually generated years ahead
rather than sized to the data.

`dim_course` at 24 and `dim_program` at 8 are the two tables that worksheet 04's
snowflake would have split into three. 37 rows against 32; the entire
star-versus-snowflake storage argument, at this scale, is five rows.

**This table is worth generating as part of the load and keeping.** Slide 41 asks
for *"table dependencies and load order"* in the specification; a row count per
dimension, per run, is the artefact that proves the load did what the
specification says — and it is the first thing to look at when a number moves.

### Question 10

Finally, build a broken dimension. Create `dim_course` from a source that has a duplicated `course_id`, join a fact spine to it with `validate="many_to_one"`. **This is supposed to fail.**

In [ ]:
broken = pd.concat([crs[["course_id", "course_name"]],
                    crs[crs.course_id == 214][["course_id", "course_name"]]],
                   ignore_index=True)
print("rows in the broken dimension:", len(broken))
print("distinct course_id:          ", broken.course_id.nunique())
print("duplicated:                  ", int(broken.course_id.duplicated().sum()))
print()
naive = enr[["enrl_id", "course_id"]].merge(broken, on="course_id", how="left")
print("fact spine rows:", len(enr), "-> after joining the broken dim:", len(naive))
print()
print(enr[["enrl_id", "course_id"]].merge(
    broken, on="course_id", how="left", validate="many_to_one").shape)

```
rows in the broken dimension: 25
distinct course_id:           24
duplicated:                   1

fact spine rows: 2400 -> after joining the broken dim: 2483

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  course_id
       214 ...
```

**One duplicated row in a 24-row dimension adds 83 rows to the fact table.**

That is course 214, which has 83 enrollments. Each of them matches two dimension
rows now, so each comes out of the join twice. 2,400 becomes 2,483 — a **3.5%**
inflation, and every measure summed over that table is 3.5% too large.

Look at the line above the error. The naive join **succeeded**: no exception, no
warning, a DataFrame with 2,483 rows and no nulls. Every internal check passes.
`enrl_id` is no longer unique, so a grain check would catch it — but only if
someone runs one, and worksheet 07 question 10 showed that grain checks pass on
tables that are wrong in other ways, which breeds complacency.

`validate="many_to_one"` catches it **before** producing a single row, and names
the offending key.

The rule this establishes:

> **Every fact-to-dimension join is `many_to_one`. Say so.**

```python
fact.merge(dim_course, on="course_id", validate="many_to_one")
```

In SQL the equivalent guarantee is a `PRIMARY KEY` or `UNIQUE` constraint on the
dimension — and Snowflake **does not enforce** declared keys, so on that platform
the check has to be an explicit assertion in the load. Worksheet 08's question 8
table is that assertion, run once per dimension, before any fact touches them.

**What this sheet established:**

| | |
|---|---|
| the dimension load | select, rename, dedupe, key — four lines per table |
| surrogate key stability | one different `ORDER BY` and cohort 301 goes from key **1 to 16**, silently repointing every historical fact |
| `dim_program` | `Unknown` category on row 8, carrying **316 enrollments** into visibility |
| `dim_student` | 606 → 606 across the city join; **7** students defaulted to `Unknown` geography |
| `dim_date` | **675 rows for 650 enrollment dates** — the 25 empty days are data |
| Unknown members | one row each, catching **7** orphan students and **1,019** no-promotion enrollments |
| the pre-flight check | 6 dimensions, PK unique, source key unique, zero nulls |
| a duplicated dimension key | 2,400 → **2,483** fact rows, with no error at all |

Worksheet 09 loads `fact_enrollment` against these six dimensions.